In [1]:
from passwords import *
from pprint import pprint
import pandas as pd
import pyodbc
import os
import boto3
import datetime as dt

In [2]:
print(f'Latest run date: {dt.datetime.today()}')

Latest run date: 2024-03-07 12:47:56.610898


### Functions

In [3]:
# upload to s3
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=None,
    )
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

### Constants

In [4]:
# project
str_project = os.getcwd().split('\\')[4].replace('_','-')
print(f'Project: {str_project}')
# task
str_task = os.getcwd().split('\\')[5]
print(f'Task: {str_task}')
# sub task
str_subtask = os.getcwd().split('\\')[6]
print(f'Subtask: {str_subtask}')
# output
str_dirname_output = './output'

Project: 20231010-gen-xii
Task: 08_retro_scoring
Subtask: 01_get_requests_from_db


### Output directory

In [5]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

### Read query

In [6]:
str_filepath = './sql/query.sql'
str_query = open(str_filepath, 'r').read()
pprint(str_query)

('select \n'
 '\ttbltempstaticpool.bigAccountId,\n'
 '\ttbltempstaticpool.dtmFunded,\n'
 '\ttblDove.strRequest\n'
 'from riskdb.analytics.tbltempstaticpool\n'
 'left outer join\n'
 '(\n'
 '\tselect\n'
 '\t\tbigAccountId,\n'
 '\t\tmax(bigDoveId) as bigDoveId\n'
 '\tfrom zestdb.dbo.tblDove\n'
 '\tgroup by bigAccountId\n'
 ')tblMax1 on tblMax1.bigAccountid=tbltempstaticpool.bigAccountId\n'
 'left outer join zestdb.dbo.tblDove on tblDove.bigDoveId=tblMax1.bigDoveId\n'
 "where dtmFunded >= '01/11/2021'")


### Write into df

In [7]:
%%time

# create connection
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)
df = pd.read_sql_query(
    str_query, 
    con=conn,
)
# close
conn.close()

# rm None requests
df = df[df['strRequest'].notna()]

# show
df

Wall time: 30min 26s


,bigAccountId,dtmFunded,strRequest
4713,5709029,2021-07-27,"{""request_id"":""589151"",""rows"":[{""row_id"":""5709..."
4718,5712734,2021-07-27,"{""request_id"":""589440"",""rows"":[{""row_id"":""5712..."
4729,5709088,2021-07-29,"{""request_id"":""589470"",""rows"":[{""row_id"":""5709..."
4742,5718748,2021-07-28,"{""request_id"":""589569"",""rows"":[{""row_id"":""5718..."
4755,5717887,2021-07-27,"{""request_id"":""589609"",""rows"":[{""row_id"":""5717..."
...,...,...,...
69710,7372352,2023-11-30,"{""request_id"":""7372352586908"",""rows"":[{""row_id..."
69711,7107327,2023-12-21,"{""request_id"":""7107327371257"",""rows"":[{""row_id..."
69712,7351212,2023-12-04,"{""request_id"":""7351212932709"",""rows"":[{""row_id..."
69713,7358983,2023-12-04,"{""request_id"":""7358983628454"",""rows"":[{""row_id..."


### Remove duplicate account ID rows (keep last)

In [8]:
df.drop_duplicates(
    subset=['bigAccountId'],
    keep='last', 
    inplace=True,
)
# show
df

,bigAccountId,dtmFunded,strRequest
4713,5709029,2021-07-27,"{""request_id"":""589151"",""rows"":[{""row_id"":""5709..."
4718,5712734,2021-07-27,"{""request_id"":""589440"",""rows"":[{""row_id"":""5712..."
4729,5709088,2021-07-29,"{""request_id"":""589470"",""rows"":[{""row_id"":""5709..."
4742,5718748,2021-07-28,"{""request_id"":""589569"",""rows"":[{""row_id"":""5718..."
4755,5717887,2021-07-27,"{""request_id"":""589609"",""rows"":[{""row_id"":""5717..."
...,...,...,...
69710,7372352,2023-11-30,"{""request_id"":""7372352586908"",""rows"":[{""row_id..."
69711,7107327,2023-12-21,"{""request_id"":""7107327371257"",""rows"":[{""row_id..."
69712,7351212,2023-12-04,"{""request_id"":""7351212932709"",""rows"":[{""row_id..."
69713,7358983,2023-12-04,"{""request_id"":""7358983628454"",""rows"":[{""row_id..."


### Save as parquet

In [9]:
%%time

# save
str_filename = 'df_requests.gzip'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_parquet(str_local_path, compression='gzip')

Wall time: 4min 33s


### Upload to s3

In [10]:
%%time

# upload
upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=f'{str_task}/{str_subtask}/{str_filename}', 
    str_bucket_name=str_project,
)

Wall time: 20.6 s


### Clean-up

In [11]:
os.remove(str_local_path)